In [2]:
import pandas as pd
import mysql.connector
from typing import Optional, List

In [24]:
def get_revenue_pivot_by_report_date(
    db_info: dict,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    tickers: Optional[List[str]] = None,
    quarter_type: Optional[str] = None
) -> pd.DataFrame:
    """
    DART 데이터에서 매출액 데이터를 추출하여 pivot 테이블 생성
    index: report_date, columns: ticker, values: thstrm_amount

    Parameters:
    -----------
    db_info : dict
        데이터베이스 연결 정보
    start_date : str, optional
        시작 날짜 (형식: 'YYYY-MM-DD')
    end_date : str, optional
        종료 날짜 (형식: 'YYYY-MM-DD')
    tickers : List[str], optional
        특정 ticker 리스트 (6자리 코드)
    quarter_type : str, optional
        분기 타입 ('Q1', 'Q3', 'H1', 'FY')

    Returns:
    --------
    pd.DataFrame
        index=report_date, columns=ticker인 pivot 테이블
    """

    conn = mysql.connector.connect(**db_info)

    try:
        # SQL 쿼리
        query = """
        SELECT
            report_date,
            ticker,
            quarter,
            account_id,
            thstrm_amount
        FROM korea_fs_data_from_DART
        WHERE account_id IN ('ifrs_Revenue', 'ifrs-full_Revenue')
          AND thstrm_amount IS NOT NULL
          AND report_date IS NOT NULL
        """

        params = []

        # 날짜 필터
        if start_date:
            query += " AND report_date >= %s"
            params.append(start_date)
        if end_date:
            query += " AND report_date <= %s"
            params.append(end_date)

        # ticker 필터
        if tickers:
            placeholders = ','.join(['%s'] * len(tickers))
            query += f" AND ticker IN ({placeholders})"
            params.extend(tickers)

        # 분기 필터
        if quarter_type:
            query += " AND quarter = %s"
            params.append(quarter_type)

        query += " ORDER BY report_date, ticker"

        # 쿼리 실행
        df = pd.read_sql(query, conn, params=params if params else None)

        if len(df) == 0:
            print("조건에 맞는 데이터가 없습니다.")
            return pd.DataFrame()

        print(f"\n{'='*60}")
        print(f"원본 데이터: {len(df):,}행")
        print(f"기간: {df['report_date'].min()} ~ {df['report_date'].max()}")
        print(f"종목(ticker) 수: {df['ticker'].nunique()}개")
        print(f"리포트 날짜 수: {df['report_date'].nunique()}개")
        print(f"{'='*60}\n")

        # 같은 report_date-ticker에 여러 account_id가 있는 경우 평균 사용
        # (ifrs_Revenue와 ifrs-full_Revenue가 둘 다 있는 경우)
        df_agg = df.groupby(['report_date', 'ticker'])['thstrm_amount'].mean().reset_index()

        # pivot 테이블 생성
        pivot_df = df_agg.pivot(index='report_date', columns='ticker', values='thstrm_amount')

        # 인덱스를 날짜 형식으로 변환
        pivot_df.index = pd.to_datetime(pivot_df.index)

        # 날짜순 정렬
        pivot_df = pivot_df.sort_index()

        print(f"Pivot 테이블 생성 완료:")
        print(f"  - 행(report_date): {len(pivot_df)}개")
        print(f"  - 열(ticker): {len(pivot_df.columns)}개")
        print(f"  - 전체 셀 수: {len(pivot_df) * len(pivot_df.columns):,}개")
        print(f"  - 결측값: {pivot_df.isna().sum().sum():,}개")
        print(f"  - 결측값 비율: {pivot_df.isna().sum().sum() / (len(pivot_df) * len(pivot_df.columns)) * 100:.2f}%\n")

        return pivot_df

    finally:
        conn.close()


def get_revenue_pivot_summary(
    db_info: dict,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    tickers: Optional[List[str]] = None,
    quarter_type: Optional[str] = None,
    fill_method: Optional[str] = None
) -> pd.DataFrame:
    """
    매출액 pivot 테이블 생성 + 기초 통계 및 결측값 처리

    Parameters:
    -----------
    db_info : dict
        데이터베이스 연결 정보
    start_date : str, optional
        시작 날짜
    end_date : str, optional
        종료 날짜
    tickers : List[str], optional
        특정 ticker 리스트
    quarter_type : str, optional
        분기 타입
    fill_method : str, optional
        결측값 처리 방법 ('ffill', 'bfill', None)

    Returns:
    --------
    pd.DataFrame
        pivot 테이블 (결측값 처리 적용)
    """

    # 기본 pivot 생성
    pivot_df = get_revenue_pivot_by_report_date(
        db_info=db_info,
        start_date=start_date,
        end_date=end_date,
        tickers=tickers,
        quarter_type=quarter_type
    )

    if len(pivot_df) == 0:
        return pivot_df

    # 결측값 처리
    if fill_method == 'ffill':
        pivot_df = pivot_df.fillna(method='ffill')
        print("결측값을 forward fill로 처리했습니다.")
    elif fill_method == 'bfill':
        pivot_df = pivot_df.fillna(method='bfill')
        print("결측값을 backward fill로 처리했습니다.")

    # 기초 통계
    print("\n=== 기초 통계 (ticker별 평균 매출액) ===")
    ticker_avg = pivot_df.mean().sort_values(ascending=False)
    print(ticker_avg.head(10))

    return pivot_df


def analyze_revenue_growth(
    pivot_df: pd.DataFrame,
    ticker: str,
    periods: int = 4
) -> pd.DataFrame:
    """
    특정 ticker의 매출액 성장률 분석

    Parameters:
    -----------
    pivot_df : pd.DataFrame
        매출액 pivot 테이블
    ticker : str
        분석할 ticker
    periods : int
        성장률 계산 기간 (분기 수)

    Returns:
    --------
    pd.DataFrame
        매출액 및 성장률 데이터프레임
    """

    if ticker not in pivot_df.columns:
        print(f"Ticker {ticker}가 데이터에 없습니다.")
        return pd.DataFrame()

    # 해당 ticker 데이터 추출
    revenue = pivot_df[ticker].dropna()

    # 성장률 계산
    result = pd.DataFrame({
        'report_date': revenue.index,
        'revenue': revenue.values,
        f'growth_{periods}q': revenue.pct_change(periods=periods) * 100
    })

    result.set_index('report_date', inplace=True)

    print(f"\n=== Ticker {ticker} 매출액 분석 ===")
    print(f"기간: {result.index.min().strftime('%Y-%m-%d')} ~ {result.index.max().strftime('%Y-%m-%d')}")
    print(f"평균 매출액: {result['revenue'].mean():,.0f}")
    print(f"최근 매출액: {result['revenue'].iloc[-1]:,.0f}")
    print(f"평균 {periods}분기 성장률: {result[f'growth_{periods}q'].mean():.2f}%")
    print(f"\n최근 5개 데이터:")
    print(result.tail())

    return result

def get_indicator_pivot(
    db_info: dict,
    indicator: str,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    tickers: Optional[List[str]] = None
) -> pd.DataFrame:
    """
    특정 indicator의 값을 pivot 테이블로 생성
    index: date, columns: ticker, values: value

    Parameters:
    -----------
    db_info : dict
        데이터베이스 연결 정보
    indicator : str
        지표명 (예: 'ETS', 'Ensemble', 'SARIMA', 'Theta')
    start_date : str, optional
        시작 날짜 (형식: 'YYYY-MM-DD')
    end_date : str, optional
        종료 날짜 (형식: 'YYYY-MM-DD')
    tickers : List[str], optional
        특정 ticker 리스트

    Returns:
    --------
    pd.DataFrame
        index=date, columns=ticker인 pivot 테이블
    """

    conn = mysql.connector.connect(**db_info)

    try:
        # 테이블명을 확인해야 합니다 (예시에서는 forecast_results로 가정)
        query = """
        SELECT
            date,
            ticker,
            indicator,
            value
        FROM korea_revenue_forecast_result
        WHERE indicator = %s
          AND value IS NOT NULL
        """

        params = [indicator]

        # 날짜 필터
        if start_date:
            query += " AND date >= %s"
            params.append(start_date)
        if end_date:
            query += " AND date <= %s"
            params.append(end_date)

        # ticker 필터
        if tickers:
            placeholders = ','.join(['%s'] * len(tickers))
            query += f" AND ticker IN ({placeholders})"
            params.extend(tickers)

        query += " ORDER BY date, ticker"

        # 쿼리 실행
        df = pd.read_sql(query, conn, params=params)

        if len(df) == 0:
            print(f"조건에 맞는 데이터가 없습니다. (indicator: {indicator})")
            return pd.DataFrame()

        print(f"\n{'='*60}")
        print(f"지표(Indicator): {indicator}")
        print(f"원본 데이터: {len(df):,}행")
        print(f"기간: {df['date'].min()} ~ {df['date'].max()}")
        print(f"종목(ticker) 수: {df['ticker'].nunique()}개")
        print(f"날짜 수: {df['date'].nunique()}개")
        print(f"{'='*60}\n")

        # 같은 date-ticker에 여러 값이 있는 경우 평균 사용 (일반적으로 없어야 함)
        df_agg = df.groupby(['date', 'ticker'])['value'].mean().reset_index()

        # pivot 테이블 생성
        pivot_df = df_agg.pivot(index='date', columns='ticker', values='value')

        # 인덱스를 날짜 형식으로 변환
        pivot_df.index = pd.to_datetime(pivot_df.index)

        # 날짜순 정렬
        pivot_df = pivot_df.sort_index()

        print(f"Pivot 테이블 생성 완료:")
        print(f"  - 행(date): {len(pivot_df)}개")
        print(f"  - 열(ticker): {len(pivot_df.columns)}개")
        print(f"  - 전체 셀 수: {len(pivot_df) * len(pivot_df.columns):,}개")
        print(f"  - 결측값: {pivot_df.isna().sum().sum():,}개")
        print(f"  - 결측값 비율: {pivot_df.isna().sum().sum() / (len(pivot_df) * len(pivot_df.columns)) * 100:.2f}%\n")

        return pivot_df

    finally:
        conn.close()


def get_multiple_indicators_pivot(
    db_info: dict,
    indicators: List[str],
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    tickers: Optional[List[str]] = None
) -> dict:
    """
    여러 indicator의 pivot 테이블을 한 번에 생성

    Parameters:
    -----------
    db_info : dict
        데이터베이스 연결 정보
    indicators : List[str]
        지표명 리스트 (예: ['ETS', 'Ensemble', 'SARIMA', 'Theta'])
    start_date : str, optional
        시작 날짜
    end_date : str, optional
        종료 날짜
    tickers : List[str], optional
        특정 ticker 리스트

    Returns:
    --------
    dict
        {indicator: pivot_df} 형태의 딕셔너리
    """

    result = {}

    for indicator in indicators:
        print(f"\n{'#'*60}")
        print(f"# {indicator} 처리 중...")
        print(f"{'#'*60}")

        pivot_df = get_indicator_pivot(
            db_info=db_info,
            indicator=indicator,
            start_date=start_date,
            end_date=end_date,
            tickers=tickers
        )

        result[indicator] = pivot_df

    return result


def compare_indicators_for_ticker(
    db_info: dict,
    ticker: str,
    indicators: List[str],
    start_date: Optional[str] = None,
    end_date: Optional[str] = None
) -> pd.DataFrame:
    """
    특정 ticker에 대해 여러 indicator 값을 비교

    Parameters:
    -----------
    db_info : dict
        데이터베이스 연결 정보
    ticker : str
        종목 코드
    indicators : List[str]
        지표명 리스트
    start_date : str, optional
        시작 날짜
    end_date : str, optional
        종료 날짜

    Returns:
    --------
    pd.DataFrame
        index=date, columns=indicator인 비교 테이블
    """

    conn = mysql.connector.connect(**db_info)

    try:
        query = """
        SELECT
            date,
            ticker,
            indicator,
            value
        FROM forecast_results
        WHERE ticker = %s
          AND value IS NOT NULL
        """

        params = [ticker]

        if start_date:
            query += " AND date >= %s"
            params.append(start_date)
        if end_date:
            query += " AND date <= %s"
            params.append(end_date)

        if indicators:
            placeholders = ','.join(['%s'] * len(indicators))
            query += f" AND indicator IN ({placeholders})"
            params.extend(indicators)

        query += " ORDER BY date, indicator"

        df = pd.read_sql(query, conn, params=params)

        if len(df) == 0:
            print(f"Ticker {ticker}에 대한 데이터가 없습니다.")
            return pd.DataFrame()

        # pivot: date x indicator
        pivot_df = df.pivot(index='date', columns='indicator', values='value')
        pivot_df.index = pd.to_datetime(pivot_df.index)
        pivot_df = pivot_df.sort_index()

        print(f"\n{'='*60}")
        print(f"Ticker {ticker} - Indicator 비교")
        print(f"기간: {pivot_df.index.min().strftime('%Y-%m-%d')} ~ {pivot_df.index.max().strftime('%Y-%m-%d')}")
        print(f"Indicators: {list(pivot_df.columns)}")
        print(f"{'='*60}\n")

        return pivot_df

    finally:
        conn.close()


def get_indicator_statistics(
    pivot_df: pd.DataFrame,
    ticker: str
) -> pd.Series:
    """
    특정 ticker의 통계 정보 계산

    Parameters:
    -----------
    pivot_df : pd.DataFrame
        pivot 테이블
    ticker : str
        종목 코드

    Returns:
    --------
    pd.Series
        통계 정보
    """

    if ticker not in pivot_df.columns:
        print(f"Ticker {ticker}가 데이터에 없습니다.")
        return pd.Series()

    stats = pivot_df[ticker].describe()

    print(f"\n=== Ticker {ticker} 통계 ===")
    print(stats)
    print(f"\n최근 5개 값:")
    print(pivot_df[ticker].tail())

    return stats


In [19]:
from DATA.stock_invest_function import fetch_table_data, get_db_host

# 데이터베이스 연결 정보
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'
}


# 예시 1: 전체 매출액 pivot 생성
print("\n" + "="*80)
print("예시 1: 전체 매출액 pivot 테이블")
print("="*80)
pivot_all = get_revenue_pivot_by_report_date(
    db_info=db_info,
    start_date='2020-01-01',
    end_date='2025-09-30'
)
print(pivot_all.head(10))
print(f"\n데이터 형태: {pivot_all.shape}")


예시 1: 전체 매출액 pivot 테이블

원본 데이터: 44,216행
기간: 2020-03-31 ~ 2025-09-30
종목(ticker) 수: 2366개
리포트 날짜 수: 23개

Pivot 테이블 생성 완료:
  - 행(report_date): 23개
  - 열(ticker): 2366개
  - 전체 셀 수: 54,418개
  - 결측값: 10,202개
  - 결측값 비율: 18.75%

ticker             000020        000040        000050        000070  \
report_date                                                           
2020-03-31   6.700367e+10  2.855757e+10  7.399416e+10  5.973060e+11   
2020-06-30   6.983330e+10  3.120514e+10  8.620966e+10  5.794880e+11   
2020-09-30   6.656821e+10  2.908280e+10  8.432171e+10  6.450780e+11   
2020-12-31   2.720754e+11  1.178344e+11  3.292247e+11  2.471226e+12   
2021-03-31   7.182724e+10  2.270011e+10  8.257075e+10  7.006650e+11   
2021-06-30   7.580235e+10  3.799214e+10  9.396038e+10  7.852480e+11   
2021-09-30   6.907114e+10  3.450467e+10  9.347672e+10  8.214320e+11   
2021-12-31   2.930181e+11  1.335104e+11  3.833571e+11  3.107313e+12   
2022-03-31   8.529413e+10  2.915054e+10  9.678620e+10  8.222970e+11

In [20]:
pivot_all[['000660', '005930']].tail(6)

ticker,000660,005930
report_date,,
2024-06-30,1.642326e+13,7.406830e+13
2024-09-30,1.757307e+13,7.909873e+13
2024-12-31,6.619296e+13,3.008709e+14
2025-03-31,1.763914e+13,7.914050e+13
2025-06-30,2.223195e+13,7.456632e+13
2025-09-30,2.444893e+13,8.606175e+13


In [25]:

# 예시 1: 단일 indicator pivot 생성
print("\n" + "="*80)
print("예시 1: ETS indicator pivot 테이블")
print("="*80)
pivot_ets = get_indicator_pivot(
    db_info=db_info,
    indicator='ETS',
    start_date='2025-01-01',
    end_date='2027-12-31'
)
print(pivot_ets.head(10))
print(f"\n데이터 형태: {pivot_ets.shape}")

# # 예시 2: 특정 ticker만 조회
# print("\n" + "="*80)
# print("예시 2: 특정 ticker의 SARIMA 예측")
# print("="*80)
# pivot_sarima = get_indicator_pivot(
#     db_info=db_info,
#     indicator='SARIMA',
#     start_date='2025-01-01',
#     tickers=['000020', '000040']
# )
# print(pivot_sarima)
#
# # 예시 3: 여러 indicator 동시 조회
# print("\n" + "="*80)
# print("예시 3: 여러 indicator 동시 조회")
# print("="*80)
# indicators = ['ETS', 'Ensemble', 'SARIMA', 'Theta']
# pivots = get_multiple_indicators_pivot(
#     db_info=db_info,
#     indicators=indicators,
#     start_date='2025-01-01',
#     end_date='2027-12-31',
#     tickers=['000020']
# )
#
# # 각 indicator별 결과 확인
# for indicator, pivot_df in pivots.items():
#     print(f"\n{indicator} 결과:")
#     print(pivot_df.head())
#
# # 예시 4: 특정 ticker에 대한 indicator 비교
# print("\n" + "="*80)
# print("예시 4: Ticker 000020의 모든 indicator 비교")
# print("="*80)
# comparison = compare_indicators_for_ticker(
#     db_info=db_info,
#     ticker='000020',
#     indicators=['ETS', 'Ensemble', 'SARIMA', 'Theta'],
#     start_date='2025-01-01'
# )
# print(comparison.head(10))
#
# # 예시 5: 통계 분석
# print("\n" + "="*80)
# print("예시 5: 통계 분석")
# print("="*80)
# if '000020' in pivot_ets.columns:
#     stats = get_indicator_statistics(pivot_ets, '000020')
#
# # 예시 6: CSV 저장
# print("\n" + "="*80)
# print("예시 6: CSV 저장")
# print("="*80)
# pivot_ets.to_csv('forecast_ets_pivot.csv', encoding='utf-8-sig')
# print("저장 완료: forecast_ets_pivot.csv")
#
# # 여러 indicator를 각각 저장
# for indicator, pivot_df in pivots.items():
#     filename = f'forecast_{indicator.lower()}_pivot.csv'
#     pivot_df.to_csv(filename, encoding='utf-8-sig')
#     print(f"저장 완료: {filename}")
#
# # 예시 7: 특정 날짜의 예측값 비교
# print("\n" + "="*80)
# print("예시 7: 특정 날짜의 예측값 Top 10")
# print("="*80)
# if len(pivot_ets) > 0:
#     target_date = pivot_ets.index[10]  # 11번째 날짜
#     forecast_values = pivot_ets.loc[target_date].dropna().sort_values(ascending=False)
#     print(f"\n{target_date.strftime('%Y-%m-%d')} 예측값 Top 10:")
#     print(forecast_values.head(10))
#
# # 예시 8: 특정 ticker의 시계열 추출
# print("\n" + "="*80)
# print("예시 8: 특정 ticker 시계열 데이터")
# print("="*80)
# if '000020' in pivot_ets.columns:
#     timeseries = pivot_ets['000020'].dropna()
#     print(f"\nTicker 000020 - ETS 예측:")
#     print(timeseries)
#     print(f"\n평균: {timeseries.mean():,.2f}")
#     print(f"표준편차: {timeseries.std():,.2f}")


예시 1: ETS indicator pivot 테이블

지표(Indicator): ETS
원본 데이터: 14,481행
기간: 2025-06-30 ~ 2027-12-31
종목(ticker) 수: 1610개
날짜 수: 38개

Pivot 테이블 생성 완료:
  - 행(date): 38개
  - 열(ticker): 1610개
  - 전체 셀 수: 61,180개
  - 결측값: 46,699개
  - 결측값 비율: 76.33%

ticker      000020  000040  000050  000070  000080  000100  000120  000140  \
date                                                                         
2025-06-30     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-01     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-02     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-03     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-04     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-05     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-06     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-07     NaN     NaN     NaN     NaN     NaN     NaN  

In [31]:
pivot_ets

ticker,000020,000040,000050,000070,000080,000100,000120,000140,000150,000180,...,226320,234080,237690,237750,244920,252500,260660,277410,355150,950130
date,,,,,,,,,,,,,,,,,,,,,
2025-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
